# Notebook 6: Model Training
**Project:** Car Price Estimation  
**Goal:** Train 3 models — Linear Regression (+ Ridge/Lasso), Random Forest, and XGBoost — with cross-validation and hyperparameter tuning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
    print('XGBoost available!')
except ImportError:
    HAS_XGB = False
    print('XGBoost not installed; using GradientBoostingRegressor instead.')

sns.set_style('whitegrid')
RANDOM_STATE = 42

# Load preprocessed data
X_train = np.load('../data/X_train.npy')
X_test  = np.load('../data/X_test.npy')
y_train = np.load('../data/y_train.npy')
y_test  = np.load('../data/y_test.npy')

feature_names = pd.read_csv('../data/feature_names.csv').iloc[:,0].tolist()

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}, y_test: {y_test.shape}')

## Helper: Evaluation Function

In [ ]:
def evaluate_model(name, model, X_train, y_train, X_test, y_test, cv=5):
    """Train, cross-validate, and evaluate a model."""
    # Cross-validation
    kf = KFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(model, X_train, y_train, cv=kf, scoring='r2')
    
    # Final fit
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    y_pred_test  = model.predict(X_test)
    
    results = {
        'Model'       : name,
        'CV_R2_Mean'  : cv_scores.mean(),
        'CV_R2_Std'   : cv_scores.std(),
        'Train_R2'    : r2_score(y_train, y_pred_train),
        'Test_R2'     : r2_score(y_test, y_pred_test),
        'Test_MAE'    : mean_absolute_error(y_test, y_pred_test),
        'Test_RMSE'   : np.sqrt(mean_squared_error(y_test, y_pred_test)),
        'y_pred_test' : y_pred_test
    }
    
    print(f'\n{"="*55}')
    print(f'  {name}')
    print(f'{"="*55}')
    print(f'  CV R² (mean ± std): {results["CV_R2_Mean"]:.4f} ± {results["CV_R2_Std"]:.4f}')
    print(f'  Train R²          : {results["Train_R2"]:.4f}')
    print(f'  Test  R²          : {results["Test_R2"]:.4f}')
    print(f'  Test  MAE         : ${results["Test_MAE"]:,.2f}')
    print(f'  Test  RMSE        : ${results["Test_RMSE"]:,.2f}')
    
    return model, results

## Model 1: Linear Regression + Ridge + Lasso

In [ ]:
# Baseline Linear Regression
lr_model, lr_results = evaluate_model(
    'Linear Regression', LinearRegression(),
    X_train, y_train, X_test, y_test
)
joblib.dump(lr_model, '../models/linear_regression.pkl')

In [ ]:
# Ridge with GridSearchCV
ridge_params = {'alpha': [0.01, 0.1, 1, 10, 100, 500]}
ridge_grid = GridSearchCV(Ridge(random_state=RANDOM_STATE), ridge_params, cv=5, scoring='r2', n_jobs=-1)
ridge_grid.fit(X_train, y_train)
print(f'Best Ridge alpha: {ridge_grid.best_params_["alpha"]}')
ridge_model, ridge_results = evaluate_model(
    'Ridge Regression', ridge_grid.best_estimator_,
    X_train, y_train, X_test, y_test
)
joblib.dump(ridge_model, '../models/ridge_regression.pkl')

In [ ]:
# Lasso with GridSearchCV
lasso_params = {'alpha': [0.01, 0.1, 1, 10, 100]}
lasso_grid = GridSearchCV(Lasso(random_state=RANDOM_STATE, max_iter=10000), lasso_params, cv=5, scoring='r2', n_jobs=-1)
lasso_grid.fit(X_train, y_train)
print(f'Best Lasso alpha: {lasso_grid.best_params_["alpha"]}')
lasso_model, lasso_results = evaluate_model(
    'Lasso Regression', lasso_grid.best_estimator_,
    X_train, y_train, X_test, y_test
)
joblib.dump(lasso_model, '../models/lasso_regression.pkl')

## Model 2: Random Forest

In [ ]:
# Random Forest with RandomizedSearchCV
from sklearn.model_selection import RandomizedSearchCV

rf_params = {
    'n_estimators'     : [100, 200, 300],
    'max_depth'        : [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4],
    'max_features'     : ['sqrt', 'log2', 0.5]
}

rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE),
    rf_params, n_iter=20, cv=5, scoring='r2',
    n_jobs=-1, random_state=RANDOM_STATE, verbose=1
)
rf_search.fit(X_train, y_train)
print(f'\nBest RF params: {rf_search.best_params_}')

rf_model, rf_results = evaluate_model(
    'Random Forest', rf_search.best_estimator_,
    X_train, y_train, X_test, y_test
)
joblib.dump(rf_model, '../models/random_forest.pkl')

In [ ]:
# Feature importance - Random Forest
importances = rf_model.feature_importances_
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(12, 7))
feat_imp.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Top 20 Feature Importances - Random Forest', fontsize=14)
ax.set_ylabel('Importance')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../reports/06_rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Model 3: XGBoost / Gradient Boosting

In [ ]:
if HAS_XGB:
    xgb_params = {
        'n_estimators'  : [100, 200, 300],
        'max_depth'     : [3, 5, 7],
        'learning_rate' : [0.01, 0.05, 0.1, 0.2],
        'subsample'     : [0.7, 0.8, 1.0],
        'colsample_bytree': [0.7, 0.8, 1.0]
    }
    base_model = XGBRegressor(random_state=RANDOM_STATE, verbosity=0, tree_method='hist')
else:
    xgb_params = {
        'n_estimators'  : [100, 200],
        'max_depth'     : [3, 5],
        'learning_rate' : [0.05, 0.1],
        'subsample'     : [0.8, 1.0]
    }
    base_model = GradientBoostingRegressor(random_state=RANDOM_STATE)

xgb_search = RandomizedSearchCV(
    base_model, xgb_params, n_iter=20, cv=5, scoring='r2',
    n_jobs=-1, random_state=RANDOM_STATE, verbose=1
)
xgb_search.fit(X_train, y_train)
print(f'\nBest Boosting params: {xgb_search.best_params_}')

model_name = 'XGBoost' if HAS_XGB else 'Gradient Boosting'
xgb_model, xgb_results = evaluate_model(
    model_name, xgb_search.best_estimator_,
    X_train, y_train, X_test, y_test
)
joblib.dump(xgb_model, '../models/xgboost_model.pkl')

## Summary: All Models

In [ ]:
all_results = [lr_results, ridge_results, lasso_results, rf_results, xgb_results]
summary_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'y_pred_test'} for r in all_results])
summary_df = summary_df.set_index('Model').round(4)
summary_df.to_csv('../reports/06_model_summary.csv')
print('Model training summary:')
print(summary_df.to_string())
print('\nNotebook 6 Complete!')